# ✂️ NorCuts — Google Colab
> **Automated viral clip cutter with Arabic subtitles, AI analysis & face tracking**

---
- Make sure **Runtime → Change runtime type → GPU (T4)** is selected
- Run cells **top to bottom**
- The WebUI link will appear after the second cell runs

In [6]:
#@title 🛠️ Installation
import os
import subprocess
import shutil
from IPython.display import clear_output

# 1. Clean any previous install
print('🧹 Cleaning previous installation...')
%cd /content
if os.path.exists('NorCuts'):
    shutil.rmtree('NorCuts')

subprocess.run('git clone --depth 1 https://github.com/zakiach555/NorCuts.git NorCuts', shell=True, check=True)
%cd /content/NorCuts

print('⏳ Installing uv and system dependencies...')

# 2. Install uv + system packages
subprocess.run(['pip', 'install', 'uv'], check=True)
subprocess.run('apt update -q && apt install -y -q ffmpeg libcairo2 xvfb', shell=True, check=True)

# 3. Create virtual environment
print('⏳ Creating virtual environment...')
subprocess.run(['uv', 'venv', '.venv'], check=True)

# 4. WhisperX + base requirements
print('⏳ Installing libraries...')
for cmd in [
    'uv pip install --python .venv git+https://github.com/m-bain/whisperx.git',
    'uv pip install --python .venv -r requirements.txt',
    'uv pip install --python .venv google-genai google-generativeai yt-dlp pandas',
]:
    subprocess.run(cmd, shell=True, check=True)

# 5. Computer vision libs
print('⏳ Installing vision dependencies...')
for cmd in [
    'uv pip install --python .venv onnxruntime-gpu insightface',
    'uv pip install --python .venv transformers==4.46.3 accelerate>=0.26.0',
]:
    subprocess.run(cmd, shell=True, check=True)

# 6. Pin Torch to stable CUDA 12.1 build
print('🔨 Pinning Torch 2.3.1...')
subprocess.run(
    'uv pip install --python .venv '
    'torch==2.3.1+cu121 torchvision==0.18.1+cu121 torchaudio==2.3.1+cu121 '
    '--index-url https://download.pytorch.org/whl/cu121',
    shell=True, check=True
)

# 7. Pin NumPy
subprocess.run("uv pip install --python .venv 'numpy<2.0' setuptools==69.5.1", shell=True, check=True)

# 8. MediaPipe (clean reinstall)
print('🔨 Fixing MediaPipe...')
subprocess.run('uv pip uninstall --python .venv -y mediapipe protobuf flatbuffers', shell=True)
subprocess.run("uv pip install --python .venv 'mediapipe>=0.10.0' 'protobuf>=3.20,<5.0' 'flatbuffers>=2.0'", shell=True, check=True)

# 9. Virtual display
os.system('Xvfb :1 -screen 0 2560x1440x8 &')
os.environ['DISPLAY'] = ':1.0'

clear_output()
print('✅ Installation complete!')
print('- Torch 2.3.1 + CUDA 12.1 : READY')
print('- Transformers 4.46.3      : READY')
print('- libcairo2 (Arabic font)  : READY')
print('\nNow run the next cell to launch the WebUI.')

🧹 Cleaning previous installation...
[WinError 2] The system cannot find the file specified: '/content'
c:\Users\ABDENNOUR ACHOUR\Documents\ViralCutter-main


PermissionError: [WinError 5] Access is denied: 'NorCuts\\.git\\objects\\pack\\pack-a29649f51d6d3a46c40038decc72d156f0eb810d.idx'

In [ ]:
#@title 🍪 Upload YouTube Cookies (Required to bypass bot detection)
# Export cookies from your browser using the extension:
# Chrome/Edge: "Get cookies.txt LOCALLY" → export for youtube.com
# Firefox:     "cookies.txt" extension → export for youtube.com
# Then upload the file below.

from google.colab import files
import shutil

print("Upload your YouTube cookies.txt file:")
uploaded = files.upload()

for filename in uploaded:
    dest = '/content/NorCuts/cookies.txt'
    with open(dest, 'wb') as f:
        f.write(uploaded[filename])
    print(f"✅ Cookies saved to {dest}")

In [ ]:
#@title 🚀 Launch NorCuts WebUI
import os

%cd /content/NorCuts

os.system('Xvfb :1 -screen 0 2560x1440x8 &')
os.environ['DISPLAY'] = ':1.0'
os.environ['MPLBACKEND'] = 'Agg'
os.environ['VIRALS_LANGUAGE'] = 'en_US'

print('🚀 Starting NorCuts WebUI...')
print('⚠️  Ignore UserWarning messages — wait for the public URL below.')
print('-' * 60)

!/content/NorCuts/.venv/bin/python webui/app.py --colab